# Handwritten Character Recognition
### CNN-based recognition on MNIST (digits) & EMNIST (characters)
**Dataset format:** CSV — columns: `label, pixel0, pixel1, ..., pixel783`

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f'TensorFlow: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Load CSV Datasets

In [ ]:
# ── CONFIG ──────────────────────────────────────────
USE_DATASET = 'mnist'   # 'mnist' or 'emnist'
# ────────────────────────────────────────────────────

if USE_DATASET == 'mnist':
    TRAIN_CSV  = 'mnist_train.csv'
    TEST_CSV   = 'mnist_test.csv'
    NUM_CLASSES = 10
    CLASS_NAMES = [str(i) for i in range(10)]
    LABEL       = 'MNIST (Digits 0–9)'
else:
    TRAIN_CSV  = 'emnist_train.csv'
    TEST_CSV   = 'emnist_test.csv'
    NUM_CLASSES = 47
    CLASS_NAMES = [str(i) for i in range(10)] + list('ABCDEFGHIJKLMNOPQRSTUVWXYZabdefghnqrt')
    LABEL       = 'EMNIST Balanced (47 Classes)'

print(f'Loading {LABEL}...')

# Read CSV
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train shape : {train_df.shape}')   # (60000, 785)
print(f'Test shape  : {test_df.shape}')    # (10000, 785)
print(f'Columns     : label + {len(train_df.columns)-1} pixel columns')
train_df.head(3)

## 3. Preprocessing

In [ ]:
# Separate label and pixels
y_train = train_df['label'].values.astype(np.int32)
y_test  = test_df['label'].values.astype(np.int32)

x_train = train_df.drop('label', axis=1).values.astype('float32')
x_test  = test_df.drop('label',  axis=1).values.astype('float32')

# Normalize [0,255] → [0,1]
x_train /= 255.0
x_test  /= 255.0

# Reshape: (N, 784) → (N, 28, 28, 1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1,  28, 28, 1)

# One-hot encode
y_train_cat = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print(f'x_train : {x_train.shape}  dtype={x_train.dtype}')
print(f'x_test  : {x_test.shape}')
print(f'y_train : {y_train_cat.shape}')

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1
)
datagen.fit(x_train)
print('Augmentation ready ✓')

## 4. Exploratory Data Analysis

In [ ]:
# Sample images
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle(f'Sample Images — {LABEL}', fontsize=13, fontweight='bold')
for row in range(3):
    for col in range(10):
        idx = np.random.randint(len(x_train))
        axes[row, col].imshow(x_train[idx].squeeze(), cmap='gray')
        axes[row, col].set_title(CLASS_NAMES[y_train[idx]], fontsize=9)
        axes[row, col].axis('off')
plt.tight_layout(); plt.show()

# Class distribution from CSV label column
fig, ax = plt.subplots(figsize=(12, 4))
train_df['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Class'); ax.set_ylabel('Count')
ax.set_title('Class Distribution (Training Set)')
ax.set_xticklabels([CLASS_NAMES[int(i)] for i in ax.get_xticks()], rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

# Pixel intensity distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(x_train.flatten(), bins=50, color='mediumseagreen', edgecolor='none')
ax.set_title('Pixel Value Distribution (after normalization)')
ax.set_xlabel('Pixel Value'); ax.set_ylabel('Frequency')
plt.tight_layout(); plt.show()

## 5. CNN Model
```
Input(28×28×1)
 ├─ Conv2D(32)→BN→Conv2D(32)→MaxPool→Dropout(0.25)
 ├─ Conv2D(64)→BN→Conv2D(64)→MaxPool→Dropout(0.25)
 ├─ Conv2D(128)→BN→GlobalAvgPool
 ├─ Dense(256)→Dropout(0.5)
 └─ Dense(NUM_CLASSES, softmax)
```

In [ ]:
def build_cnn(num_classes):
    inp = keras.Input(shape=(28, 28, 1))

    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    m = keras.Model(inp, out, name='HandwrittenCNN')
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    return m

model = build_cnn(NUM_CLASSES)
model.summary()

## 6. Training
> Set `TRAIN = True` to train. Expected: MNIST ~99.3% · EMNIST ~85–88%

In [ ]:
TRAIN      = False
BATCH_SIZE = 128
EPOCHS     = 30

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)
]

if TRAIN:
    history = model.fit(
        datagen.flow(x_train, y_train_cat, batch_size=BATCH_SIZE),
        steps_per_epoch=len(x_train) // BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(x_test, y_test_cat),
        callbacks=callbacks
    )
    model.save('handwritten_cnn_model.keras')
else:
    print('Skipped. Set TRAIN=True to train.')
    # Simulated curves for visualization
    np.random.seed(0); ep = np.arange(1, 21)
    history = type('H', (), {'history': {
        'accuracy':     np.clip(1 - 0.5*np.exp(-0.2*ep)  + np.random.normal(0, 0.005, 20), 0, 1),
        'val_accuracy': np.clip(1 - 0.55*np.exp(-0.18*ep)+ np.random.normal(0, 0.007, 20), 0, 1),
        'loss':         np.clip(0.6*np.exp(-0.25*ep)      + np.random.normal(0, 0.005, 20), 0, None),
        'val_loss':     np.clip(0.65*np.exp(-0.22*ep)     + np.random.normal(0, 0.008, 20), 0, None),
    }})()

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training History', fontsize=14, fontweight='bold')
ax1.plot(history.history['accuracy'],     label='Train', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val',   lw=2, ls='--')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history['loss'],         label='Train', lw=2, color='tomato')
ax2.plot(history.history['val_loss'],     label='Val',   lw=2, ls='--', color='darkorange')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Evaluation

In [ ]:
loss, acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'Test Loss    : {loss:.4f}')
print(f'Test Accuracy: {acc*100:.2f}%')

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)

# Report on first 10 classes
mask = y_test < 10
print('\nClassification Report (first 10 classes):')
print(classification_report(y_test[mask], y_pred[mask],
                             target_names=CLASS_NAMES[:10], zero_division=0))

In [ ]:
# Confusion Matrix
n_show = min(10, NUM_CLASSES)
mask   = y_test < n_show
cm     = confusion_matrix(y_test[mask], y_pred[mask])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES[:n_show],
            yticklabels=CLASS_NAMES[:n_show], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {LABEL}')
plt.tight_layout(); plt.show()

## 9. Prediction Visualization

In [ ]:
indices = np.random.choice(len(x_test), 20, replace=False)
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Predictions  (green=correct · red=wrong)', fontsize=12)
for i, idx in enumerate(indices):
    ax = axes[i//10, i%10]
    ax.imshow(x_test[idx].squeeze(), cmap='gray')
    correct = y_pred[idx] == y_test[idx]
    ax.set_title(f'P:{CLASS_NAMES[y_pred[idx]]}\nT:{CLASS_NAMES[y_test[idx]]}',
                 color='green' if correct else 'red', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 10. Feature Map Visualization

In [ ]:
act_model   = keras.Model(inputs=model.input, outputs=model.layers[1].output)
feat_maps   = act_model.predict(x_test[0:1], verbose=0)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle(f'Conv Layer 1 Feature Maps — Label: {CLASS_NAMES[y_test[0]]}', fontsize=12)
axes[0,0].imshow(x_test[0].squeeze(), cmap='gray'); axes[0,0].set_title('Input',fontsize=8); axes[0,0].axis('off')
for i in range(1, 32):
    r, c = (i+1)//8, (i+1)%8
    if r < 4:
        axes[r,c].imshow(feat_maps[0,:,:,i], cmap='viridis')
        axes[r,c].set_title(f'F{i}', fontsize=7); axes[r,c].axis('off')
plt.tight_layout(); plt.show()

## 11. Predict Custom Image

In [ ]:
from PIL import Image, ImageOps

def predict_image(path, model, class_names):
    img = Image.open(path).convert('L')
    img = ImageOps.invert(img)
    img = img.resize((28, 28), Image.LANCZOS)
    arr = np.array(img).astype('float32') / 255.0
    arr = arr[np.newaxis, ..., np.newaxis]          # (1,28,28,1)
    probs  = model.predict(arr, verbose=0)[0]
    pred   = np.argmax(probs)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
    ax1.imshow(arr[0].squeeze(), cmap='gray')
    ax1.set_title(f'Predicted: "{class_names[pred]}" ({probs[pred]*100:.1f}%)', fontsize=12)
    ax1.axis('off')
    top5 = np.argsort(probs)[::-1][:5]
    ax2.barh([class_names[i] for i in top5][::-1], probs[top5][::-1]*100, color='steelblue')
    ax2.set_xlabel('Confidence (%)'); ax2.set_title('Top-5'); ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.show()
    return class_names[pred], probs[pred]

# Usage: predict_image('my_digit.png', model, CLASS_NAMES)
print('Ready. Call: predict_image("your_image.png", model, CLASS_NAMES)')

## 12. CRNN Extension (Word/Sentence Recognition)

In [ ]:
def build_crnn(input_shape=(32, 128, 1), num_classes=27):
    """CNN + BiLSTM + CTC for sequence (word/sentence) recognition."""
    inp = keras.Input(shape=input_shape)
    x = layers.Conv2D(64,  (3,3), padding='same', activation='relu')(inp)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)
    x = layers.Conv2D(512, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2,1))(x)
    sh = x.shape
    x  = layers.Reshape((sh[2], sh[1]*sh[3]))(x)
    x  = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x  = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    out = layers.Dense(num_classes + 1, activation='softmax')(x)  # +1 for CTC blank
    return keras.Model(inp, out, name='CRNN')

crnn = build_crnn()
crnn.summary()

## Summary

| | MNIST | EMNIST |
|---|---|---|
| **CSV Files** | `mnist_train.csv`, `mnist_test.csv` | `emnist_train.csv`, `emnist_test.csv` |
| **Rows** | 60k train / 10k test | 112.8k train / 18.8k test |
| **Columns** | label + pixel0…pixel783 (785 total) | same format |
| **Classes** | 10 (digits 0–9) | 47 (digits + letters) |
| **Expected Acc** | ~99.3% | ~85–88% |

### Load real EMNIST CSV from Kaggle:
```
https://www.kaggle.com/datasets/crawford/emnist
```
Same column format — drop in as replacement for `emnist_train.csv`.